# OCR evaluation: CRNN vs EasyOCR + ablation препроцессинга из ДЗ

**Что сравниваем:**
1. **CRNN+CTC** — наша собственная сеть (`src/crnn.py`), обученная в `02b_train_crnn_ocr.ipynb`. Читает произвольный текст.
2. **EasyOCR** baseline — готовая библиотека, uk+en.

**Ablation препроцессинга:** проверяем вклад каждого классического шага из курса (ДЗ2 gray-world, ДЗ3 unsharp, ДЗ6+7 rectify, ДЗ8 otsu) на обе модели.

**Метрики:** mean CER (character error rate), exact-match %, breakdown по форматам (standard/named/diplomat/...).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np, pandas as pd, cv2, torch
import matplotlib.pyplot as plt
import easyocr

from src.preprocess import gray_world, unsharp_mask, prep_for_ocr
from src.crnn import CRNN, NUM_CLASSES, decode_greedy, preprocess_for_crnn, CHARSET
from src.pipeline import UA_ALLOWED

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- CRNN
crnn = CRNN(num_classes=NUM_CLASSES).to(DEVICE).eval()
ckpt_path = Path('../models/crnn_ocr.pt')
if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    state = ckpt['model_state_dict'] if isinstance(ckpt, dict) and 'model_state_dict' in ckpt else ckpt
    crnn.load_state_dict(state)
    print(f'CRNN loaded: {ckpt_path}')
else:
    print(f'⚠ {ckpt_path} не найден — сначала обучите в 02b_train_crnn_ocr.ipynb')

# --- EasyOCR
reader = easyocr.Reader(['uk', 'en'], gpu=(DEVICE == 'cuda'))
print('EasyOCR ready')

## 1. Тестовый набор

In [ ]:
# Собираем test-набор: val из synthetic_ocr + real crops из AUTO.RIA (если размечены)
import csv as _csv

candidates = [
    (Path('../data/synthetic_ocr/images/val'), Path('../data/synthetic_ocr/labels_val.csv')),
    (Path('../data/ocr_crops/val'),            Path('../data/ocr_crops/labels_val.csv')),
]

samples = []
for root, csv_path in candidates:
    if not csv_path.exists():
        continue
    with open(csv_path, encoding='utf-8') as f:
        for row in _csv.DictReader(f):
            path = root / row['file']
            if path.exists():
                samples.append({
                    'path': path,
                    'text': row['text'].strip().upper(),
                    'kind': row.get('kind', 'real'),
                })

df = pd.DataFrame(samples)
print(f'всего примеров: {len(df)}')
df['kind'].value_counts() if len(df) else None

## 2. Утилиты

In [ ]:
def levenshtein(a, b):
    if len(a) < len(b): a, b = b, a
    if not b: return len(a)
    prev = list(range(len(b)+1))
    for i, ca in enumerate(a, 1):
        cur = [i] + [0]*len(b)
        for j, cb in enumerate(b, 1):
            cur[j] = min(cur[j-1]+1, prev[j]+1, prev[j-1]+(ca != cb))
        prev = cur
    return prev[-1]

def cer(pred, gt): return levenshtein(pred, gt) / max(len(gt), 1)

def easyocr_read(img, strict=False):
    allowlist = UA_ALLOWED if strict else None
    res = reader.readtext(img, allowlist=allowlist, detail=1)
    if not res: return ''
    res.sort(key=lambda r: r[0][0][0])
    return ''.join(r[1] for r in res).upper()

def crnn_read(img):
    tensor = preprocess_for_crnn(img).to(DEVICE)
    with torch.no_grad():
        log_probs = crnn(tensor)
    return decode_greedy(log_probs)[0]

## 3. Основной эксперимент: CRNN vs EasyOCR × препроцессинг-режимы

In [ ]:
MODES = {
    'raw':            dict(use_gray_world=False, use_unsharp=False, use_rectify=False, use_otsu=False),
    '+gray_world':    dict(use_gray_world=True,  use_unsharp=False, use_rectify=False, use_otsu=False),
    '+unsharp':       dict(use_gray_world=True,  use_unsharp=True,  use_rectify=False, use_otsu=False),
    '+rectify':       dict(use_gray_world=True,  use_unsharp=True,  use_rectify=True,  use_otsu=False),
    '+otsu (full)':   dict(use_gray_world=True,  use_unsharp=True,  use_rectify=True,  use_otsu=True),
}

rows = []
for _, r in df.iterrows():
    img = cv2.cvtColor(cv2.imread(str(r['path'])), cv2.COLOR_BGR2RGB)
    for mode, flags in MODES.items():
        prep = prep_for_ocr(img, **flags)
        for backend, pred in [('CRNN', crnn_read(prep)), ('EasyOCR', easyocr_read(prep))]:
            rows.append({
                'kind': r['kind'], 'mode': mode, 'backend': backend,
                'gt': r['text'], 'pred': pred,
                'cer': cer(pred, r['text']),
                'exact': int(pred == r['text']),
            })

exp = pd.DataFrame(rows)
print(f'запусков: {len(exp)}')
exp.head()

## 4. Сводка: mean CER и exact-match по backend × mode

In [ ]:
summary = exp.groupby(['backend', 'mode']).agg(
    mean_CER=('cer', 'mean'),
    exact_pct=('exact', lambda x: 100*x.mean()),
).round(3).unstack('backend')
summary

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for backend in ['CRNN', 'EasyOCR']:
    data = exp[exp['backend']==backend].groupby('mode')['cer'].mean()
    ax[0].plot(list(MODES.keys()), [data[m] for m in MODES], marker='o', label=backend)
    data = exp[exp['backend']==backend].groupby('mode')['exact'].mean() * 100
    ax[1].plot(list(MODES.keys()), [data[m] for m in MODES], marker='o', label=backend)
ax[0].set_ylabel('mean CER (ниже лучше)'); ax[0].legend(); ax[0].grid(True, alpha=0.3)
ax[1].set_ylabel('exact match, %'); ax[1].legend(); ax[1].grid(True, alpha=0.3)
for a in ax: a.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

## 5. Breakdown по форматам номеров (kind)

In [ ]:
best_mode = '+rectify'   # можно выбрать иной по результатам выше
by_kind = exp[exp['mode']==best_mode].groupby(['kind','backend']).agg(
    n=('gt','count'), mean_CER=('cer','mean'), exact_pct=('exact', lambda x: 100*x.mean())
).round(3)
by_kind

## 6. Визуальный пример — как меняется кроп по стадиям препроцессинга

In [ ]:
if len(df):
    sample = df.iloc[0]
    img = cv2.cvtColor(cv2.imread(str(sample['path'])), cv2.COLOR_BGR2RGB)
    stages = {'raw': img,
              'gray_world': gray_world(img),
              'unsharp': unsharp_mask(gray_world(img)),
              'rectify': prep_for_ocr(img, use_gray_world=True, use_unsharp=True, use_rectify=True, use_otsu=False),
              'otsu': prep_for_ocr(img, use_gray_world=True, use_unsharp=True, use_rectify=True, use_otsu=True)}
    fig, axes = plt.subplots(1, len(stages), figsize=(20, 4))
    for ax, (name, stage) in zip(axes, stages.items()):
        disp = stage if stage.ndim == 3 else cv2.cvtColor(stage, cv2.COLOR_GRAY2RGB)
        c_pred = crnn_read(stage)
        e_pred = easyocr_read(stage)
        ax.imshow(disp); ax.axis('off')
        ax.set_title(f'{name}\nGT: {sample["text"]}\nCRNN: {c_pred}\nEasy: {e_pred}', fontsize=9)
    plt.tight_layout(); plt.show()

## Выводы

Ожидаемая картина:
- **CRNN** выигрывает на именных/нестандартных форматах (EasyOCR без allowlist часто путает символы, с allowlist — режет именные)
- **Rectify+unsharp** (ДЗ3+6+7) даёт наибольший прирост exact-match на наклонённых/размытых кропах
- **Otsu** (ДЗ8) помогает на контрастных номерах, мешает на цветных (красные дипломатические, жёлтые такси)
- **Gray-world** (ДЗ2) стабильно полезен под плохое освещение